# ODRL Policy Nanopublication Generator

Creates ODRL (Open Digital Rights Language) policy nanopublications from a JSON configuration file.

Each nanopub encodes an access policy for a FAIR2Adapt dataset, specifying:
- **Permissions**: what actions are allowed (e.g., use, reproduce) and under what constraints (e.g., academic research only)
- **Prohibitions**: what actions are not allowed (e.g., distribute, commercialize)
- **Duties**: what obligations must be met (e.g., attribution)

**Template**: To be created on [Nanodash](https://nanodash.knowledgepixels.com/publish) — see instructions below.

---

## How to create the ODRL Policy template on Nanodash

Before using this notebook, create an assertion template on Nanodash:

1. Go to [Nanodash → Publish → Assertion Template](https://nanodash.knowledgepixels.com/publish?formobj=8905034205668433352&template=https%3A%2F%2Fw3id.org%2Fnp%2FFRAQIZQdqa_kqYfFSnboJ...)
2. Fill in:
   - **Name**: `ODRL Access Policy`
   - **Description**: `Defines an ODRL policy for controlling access to a FAIR dataset. Specifies permissions (with purpose constraints), prohibitions, and attribution duties.`
   - **Tag**: `FAIR2Adapt`
3. Define the statements (see the assertion structure in the generated TriG files)
4. Publish the template and copy its URI
5. Update `ODRL_POLICY_TEMPLATE` below with the URI

---

---
# 📝 SECTION 1: INPUT FILE (EDIT THIS)
---

In [ ]:
# Path to your ODRL policy JSON config file
CONFIG_FILE = "config/hamburg_odrl_policy.json"

OUTPUT_DIR = "output/"

---
# ⚙️ SECTION 2: SETUP
---

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone
from rdflib import Dataset, Namespace, URIRef, Literal, BNode
from rdflib.namespace import RDF, RDFS, XSD, FOAF

# Namespaces
NP = Namespace("http://www.nanopub.org/nschema#")
DCT = Namespace("http://purl.org/dc/terms/")
NT = Namespace("https://w3id.org/np/o/ntemplate/")
NPX = Namespace("http://purl.org/nanopub/x/")
PROV = Namespace("http://www.w3.org/ns/prov#")
ORCID = Namespace("https://orcid.org/")
ODRL = Namespace("http://www.w3.org/ns/odrl/2/")
DPV = Namespace("https://w3id.org/dpv#")

# Template URIs — UPDATE after publishing template with create_odrl_template.ipynb
ODRL_POLICY_TEMPLATE = None  # e.g., URIRef("https://w3id.org/np/RA...")
PROV_TEMPLATE = URIRef("https://w3id.org/np/RA7lSq6MuK_TIC6JMSHvLtee3lpLoZDOqLJCLXevnrPoU")
PUBINFO_TEMPLATE_1 = URIRef("https://w3id.org/np/RA0J4vUn_dekg-U1kK3AOEt02p9mT2WO03uGxLDec1jLw")
PUBINFO_TEMPLATE_2 = URIRef("https://w3id.org/np/RAukAcWHRDlkqxk7H2XNSegc1WnHI569INvNr-xdptDGI")

# ODRL action URIs
ODRL_ACTIONS = {
    "use": ODRL.use,
    "reproduce": ODRL.reproduce,
    "distribute": ODRL.distribute,
    "commercialize": ODRL.commercialize,
    "derive": ODRL.derive,
    "modify": ODRL.modify,
    "attribute": ODRL.attribute,
    "present": ODRL.present,
    "sell": ODRL.sell,
    "transfer": ODRL.transfer,
}

# Purpose URIs — W3C Data Privacy Vocabulary (DPV)
# These are valid, resolvable URIs at https://w3id.org/dpv#
PURPOSE_URIS = {
    "AcademicResearch": DPV.AcademicResearch,
    "ScientificResearch": DPV.ScientificResearch,
    "NonCommercialResearch": DPV.NonCommercialResearch,
    "PublicBenefit": DPV.PublicBenefit,
}

print("✓ Setup complete")

---
# 📖 SECTION 3: LOAD & VALIDATE
---

In [ ]:
print(f"Loading: {CONFIG_FILE}")

with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
    config = json.load(f)

metadata = config.get('metadata', {})
AUTHOR_ORCID = metadata.get('creator_orcid')
AUTHOR_NAME = metadata.get('creator_name')

errors = []
if not AUTHOR_ORCID:
    errors.append("metadata.creator_orcid is required")
if not AUTHOR_NAME:
    errors.append("metadata.creator_name is required")
if not config.get('nanopublications'):
    errors.append("nanopublications list is required")

for i, np_config in enumerate(config.get('nanopublications', [])):
    if not np_config.get('policy_uid'):
        errors.append(f"nanopublications[{i}].policy_uid is required")
    if not np_config.get('target', {}).get('uri'):
        errors.append(f"nanopublications[{i}].target.uri is required")

if errors:
    print("❌ Validation errors:")
    for e in errors:
        print(f"   - {e}")
    raise ValueError("Please fix the errors in your JSON file")

print(f"✓ Loaded {len(config['nanopublications'])} ODRL policy nanopubs to generate")
print(f"✓ Author: {AUTHOR_NAME} ({AUTHOR_ORCID})")

---
# 🔨 SECTION 4: BUILD NANOPUBLICATIONS
---

In [ ]:
def create_odrl_policy_nanopub(np_config, metadata):
    """
    Create an ODRL policy nanopublication using rdflib Dataset.
    
    The assertion graph contains the ODRL policy with:
    - Policy type (Offer/Set/Agreement)
    - Permissions with action and purpose constraints
    - Prohibitions
    - Duties (e.g., attribution)
    """
    TEMP_NP = Namespace("http://purl.org/nanopub/temp/np/")
    
    this_np = URIRef("http://purl.org/nanopub/temp/np/")
    head_graph = URIRef("http://purl.org/nanopub/temp/np/Head")
    assertion_graph = URIRef("http://purl.org/nanopub/temp/np/assertion")
    provenance_graph = URIRef("http://purl.org/nanopub/temp/np/provenance")
    pubinfo_graph = URIRef("http://purl.org/nanopub/temp/np/pubinfo")
    
    author_uri = ORCID[metadata['creator_orcid']]
    policy_uri = URIRef(np_config['policy_uid'])
    target_uri = URIRef(np_config['target']['uri'])
    policy_type = np_config.get('policy_type', 'Offer')
    
    ds = Dataset()
    
    # Bind prefixes
    ds.bind("this", "http://purl.org/nanopub/temp/np/")
    ds.bind("sub", TEMP_NP)
    ds.bind("np", NP)
    ds.bind("dct", DCT)
    ds.bind("nt", NT)
    ds.bind("npx", NPX)
    ds.bind("xsd", XSD)
    ds.bind("rdfs", RDFS)
    ds.bind("orcid", ORCID)
    ds.bind("prov", PROV)
    ds.bind("foaf", FOAF)
    ds.bind("odrl", ODRL)
    ds.bind("dpv", DPV)
    
    # HEAD graph
    head = ds.graph(head_graph)
    head.add((this_np, RDF.type, NP.Nanopublication))
    head.add((this_np, NP.hasAssertion, assertion_graph))
    head.add((this_np, NP.hasProvenance, provenance_graph))
    head.add((this_np, NP.hasPublicationInfo, pubinfo_graph))
    
    # ASSERTION graph — the ODRL policy
    assertion = ds.graph(assertion_graph)
    
    # Policy type
    assertion.add((policy_uri, RDF.type, ODRL[policy_type]))
    assertion.add((policy_uri, ODRL.uid, policy_uri))
    assertion.add((policy_uri, ODRL.profile, URIRef("http://www.w3.org/ns/odrl/2/core")))
    
    # Permissions
    for perm in np_config.get('permissions', []):
        perm_node = BNode()
        assertion.add((policy_uri, ODRL.permission, perm_node))
        assertion.add((perm_node, ODRL.target, target_uri))
        
        for action_name in perm.get('actions', []):
            action_uri = ODRL_ACTIONS.get(action_name, ODRL[action_name])
            assertion.add((perm_node, ODRL.action, action_uri))
        
        # Purpose constraint
        if perm.get('purpose'):
            constraint_node = BNode()
            assertion.add((perm_node, ODRL.constraint, constraint_node))
            assertion.add((constraint_node, ODRL.leftOperand, ODRL.purpose))
            assertion.add((constraint_node, ODRL.operator, ODRL.eq))
            # Support both short names and full DPV URIs
            purpose_val = perm['purpose']
            if purpose_val.startswith("http"):
                purpose_uri = URIRef(purpose_val)
            else:
                purpose_uri = PURPOSE_URIS.get(purpose_val, DPV[purpose_val])
            assertion.add((constraint_node, ODRL.rightOperand, purpose_uri))
        
        # Assignee
        if perm.get('assignee'):
            assertion.add((perm_node, ODRL.assignee, URIRef(perm['assignee'])))
    
    # Prohibitions
    for prohib in np_config.get('prohibitions', []):
        prohib_node = BNode()
        assertion.add((policy_uri, ODRL.prohibition, prohib_node))
        assertion.add((prohib_node, ODRL.target, target_uri))
        
        for action_name in prohib.get('actions', []):
            action_uri = ODRL_ACTIONS.get(action_name, ODRL[action_name])
            assertion.add((prohib_node, ODRL.action, action_uri))
    
    # Duties
    for duty in np_config.get('duties', []):
        duty_node = BNode()
        assertion.add((policy_uri, ODRL.duty, duty_node))
        action_uri = ODRL_ACTIONS.get(duty['action'], ODRL[duty['action']])
        assertion.add((duty_node, ODRL.action, action_uri))
        
        if duty.get('attributed_party', {}).get('uri'):
            assertion.add((duty_node, ODRL.attributedParty,
                          URIRef(duty['attributed_party']['uri'])))
    
    # PROVENANCE graph
    provenance = ds.graph(provenance_graph)
    provenance.add((assertion_graph, PROV.wasAttributedTo, author_uri))
    
    # PUBINFO graph
    pubinfo = ds.graph(pubinfo_graph)
    pubinfo.add((author_uri, FOAF.name, Literal(metadata['creator_name'])))
    
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
    pubinfo.add((this_np, DCT.created, Literal(now, datatype=XSD.dateTime)))
    pubinfo.add((this_np, DCT.creator, author_uri))
    pubinfo.add((this_np, DCT.license, URIRef("https://creativecommons.org/licenses/by/4.0/")))
    pubinfo.add((this_np, NPX.wasCreatedAt, URIRef("https://fair2adapt-eosc.eu")))
    
    # Nanopub type
    pubinfo.add((this_np, NPX.hasNanopubType, ODRL.Policy))
    
    # Introduces the policy
    pubinfo.add((this_np, NPX.introduces, policy_uri))
    
    # Label
    target_label = np_config['target'].get('label', np_config['target']['uri'])
    label = f"ODRL policy: {target_label}"
    pubinfo.add((this_np, RDFS.label, Literal(label)))
    
    # Template references
    if ODRL_POLICY_TEMPLATE:
        pubinfo.add((this_np, NT.wasCreatedFromTemplate, ODRL_POLICY_TEMPLATE))
    pubinfo.add((this_np, NT.wasCreatedFromProvenanceTemplate, PROV_TEMPLATE))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_1))
    pubinfo.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_2))
    
    return ds, label

print("✓ Function defined")

In [ ]:
# Create output directory
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Generate all nanopublications
generated_files = []

for np_config in config['nanopublications']:
    ds, label = create_odrl_policy_nanopub(np_config, metadata)
    
    output_file = Path(OUTPUT_DIR) / f"{np_config['id']}.trig"
    ds.serialize(destination=str(output_file), format='trig')
    generated_files.append(output_file)
    
    print(f"✓ Generated: {output_file}")

print(f"\nTotal generated: {len(generated_files)} nanopublications")

---
# 📄 SECTION 5: PREVIEW
---

In [ ]:
if generated_files:
    print(f"Preview of {generated_files[0]}:\n")
    print("=" * 80)
    with open(generated_files[0], 'r') as f:
        print(f.read())

---
# 🚀 SECTION 6: SIGN & PUBLISH
---

In [ ]:
PUBLISH = False  # Set to True when ready to publish
USE_TEST_SERVER = True  # Set to False for production
PROFILE_PATH = None  # Set to your profile.yml path

In [ ]:
if PUBLISH:
    from nanopub import Nanopub, NanopubConf, load_profile
    
    profile = load_profile(PROFILE_PATH)
    print(f"Loaded profile: {profile.name}")
    
    conf = NanopubConf(profile=profile, use_test_server=USE_TEST_SERVER)
    
    published_uris = []
    for trig_file in generated_files:
        np_obj = Nanopub(rdf=trig_file, conf=conf)
        np_obj.sign()
        
        signed_path = trig_file.with_suffix('.signed.trig')
        np_obj.store(signed_path)
        print(f"✓ Signed: {signed_path}")
        
        np_obj.publish()
        published_uris.append(np_obj.source_uri)
        print(f"✓ Published: {np_obj.source_uri}")
    
    # Save published URIs for updating registry.json
    print("\n" + "=" * 80)
    print("Update policies/registry.json with these nanopub URIs:")
    print("=" * 80)
    for np_config, uri in zip(config['nanopublications'], published_uris):
        dataset_id = np_config['id'].replace('odrl_', '').replace('_', '-')
        print(f'  "{dataset_id}": {{"policy_nanopub": "{uri}"}}')
else:
    print("Publishing disabled. Set PUBLISH = True to enable.")
    print("\nManual signing:")
    for f in generated_files:
        print(f"  nanopub sign {f}")
        print(f"  nanopub publish {f.stem}.signed.trig")

---
# 📋 JSON Config Template

```json
{
  "metadata": {
    "creator_orcid": "0000-0002-1784-2920",
    "creator_name": "Your Name",
    "project": {
      "uri": "https://fair2adapt-eosc.eu",
      "label": "FAIR2Adapt"
    }
  },
  "nanopublications": [
    {
      "id": "odrl_my_dataset",
      "policy_uid": "https://fair2adapt.eu/policy/my-dataset",
      "policy_type": "Offer",
      "target": {
        "uri": "https://fair2adapt.eu/data/my-dataset",
        "label": "Description of the dataset"
      },
      "permissions": [
        {
          "actions": ["use", "reproduce"],
          "purpose": "AcademicResearch"
        }
      ],
      "prohibitions": [
        { "actions": ["distribute", "commercialize"] }
      ],
      "duties": [
        {
          "action": "attribute",
          "attributed_party": {
            "uri": "https://fair2adapt-eosc.eu",
            "label": "FAIR2Adapt"
          }
        }
      ]
    }
  ]
}
```
---